# Indic Sentiment Classification: Fine-Tuned DistilBERT vs Prompted 7B LLM
### Accuracy per Dollar & Efficiency Benchmark (Hindi, Kannada, Telugu)

This notebook trains and evaluates:
1. **Fine-Tuned Multilingual DistilBERT (`distilbert-base-multilingual-cased`)** on 500, 2,000, and 10,000 samples.
2. **Zero-Shot & 5-Shot 7B LLM (`Qwen/Qwen2.5-7B-Instruct` in 4-bit)** on the exact same test set.
3. Compares Macro-F1, peak VRAM, training time, inference throughput (reviews/sec), and cost per 1,000 reviews.

## 1. Verify GPU & Install Dependencies
Ensure runtime type is set to **GPU (T4 or higher)**: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q torch transformers datasets accelerate bitsandbytes scikit-learn pandas numpy tqdm

## 2. Prepare Indic Sentiment Dataset
Downloads and formats data from Hugging Face `ai4bharat/IndicSentiment` (or creates a stratified sample) across Hindi, Kannada, and Telugu, flags code-mixed sentences, and saves stratified 80/20 train/test CSVs.

In [ ]:
# Run data preparation
!python data/prepare_data.py --train-size 10000 --test-size 2500

## 3. Fine-Tune DistilBERT Multilingual (3 Sample Sizes: 500, 2000, 10000)
Evaluates on `data/test.csv`, tracking peak VRAM (`torch.cuda.max_memory_allocated`), wall-clock time, loss, and macro-F1.

In [ ]:
# Trains all 3 runs (500, 2000, 10000) and saves results/distilbert_<size>.json
!python train_distilbert.py --train-csv data/train.csv --test-csv data/test.csv --output-dir results

## 4. Evaluate 7B LLM (Qwen2.5-7B-Instruct in 4-bit NF4)
Tests Zero-Shot and 5-Shot in-context learning with strict regex extraction on the identical test set.

In [ ]:
# Run LLM zero-shot & 5-shot evaluation
!python eval_llm.py --test-csv data/test.csv --train-csv data/train.csv --output-dir results

## 5. Aggregate Results & Generate Error Analysis Samples

In [ ]:
!python make_report.py --results-dir results

## 6. Download Results Package to Local Repository
This creates `results.zip` containing all real metric JSON files to drop directly into your local `results/` folder.

In [ ]:
from google.colab import files
!zip -r results.zip results/
files.download('results.zip')